In [ ]:
# 导入 fastbook（会一并带入 fastai 视觉相关的全部工具，如 search_images_ddg）
import numpy as np
from fastbook import *

# 用 DuckDuckGo 搜索 "bird photos"，只取 1 张图片的 URL，先验证搜索能用
urls = search_images_ddg('bird photos', max_images=1)
len(urls), urls[0]


In [ ]:
# 把第一张图片下载到本地 bird.jpg（已存在就跳过，避免重复下载）
dest = Path('bird.jpg')
if not dest.exists(): download_url(urls[0], dest, show_progress=False)


In [ ]:
# 打开图片并生成 256x256 缩略图预览
im = Image.open(dest)
im.to_thumb(256, 256)


In [ ]:
# 分别搜索 "forest" 和 "bird"，各下载约 200 张，构成二分类数据集
searches = 'forest', 'bird'
path = Path('bird_or_not')
if not path.exists():
    for o in searches:
        dest = (path/o)
        dest.mkdir(exist_ok=True, parents=True)      # parents=True：父目录 bird_or_not 不存在时一并创建
        results = search_images_ddg(f'{o} photo')    # 修正：原来是 {0}，应为循环变量 {o}
        download_images(dest, urls=results[:200])
        resize_images(dest, max_size=400, dest=dest) # 统一缩放到最大 400px，加快训练


In [ ]:
# 校验图片完整性，删除损坏 / 无法打开的文件
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)


In [ ]:
# 用 DataBlock 定义数据管道：输入图片、输出类别；8:2 划分训练/验证；用父文件夹名当标签
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,                        # 修正：补上 get_items，否则找不到图片文件
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')],
).dataloaders(path)
dls.show_batch(max_n=6)


In [ ]:
# 用预训练 resnet18 迁移学习，微调 3 轮；error_rate 越低越好
learn = cnn_learner(dls, resnet18, metrics=error_rate)   # 新版 fastai 里 cnn_learner 已更名为 vision_learner
learn.fine_tune(3)


In [ ]:
# 用训练好的模型预测 bird.jpg
is_bird, _, probs = learn.predict(PILImage.create('bird.jpg'))
print(f"this is a :{is_bird}")
print(f" probability its a bird :{probs[0]:.4f} ")


In [ ]:
# —— 图像分割示例（CamVid 数据集）——
path = untar_data(URLs.CAMVID_TINY)
dls = SegmentationDataLoaders.from_label_func(
    path, bs=8, fnames=get_image_files(path/'images'),
    label_func=lambda o: path/'labels'/f'{o.stem}_P{o.suffix}',  # 修正：标签文件名后缀是大写 _P
    codes=np.loadtxt(path/'codes.txt', dtype=str)                # 每个像素类别的名称表
)
learn = unet_learner(dls, resnet34)   # 修正：原来是 resent34 拼写错误
learn.fine_tune(8)


In [ ]:
# 展示分割预测结果
learn.show_results(max_n=3, figsize=(7, 8))


In [ ]:
# —— 表格数据示例（成年人收入预测）——
from fastai.tabular.all import *
path = untar_data(URLs.ADULT_SAMPLE)
dls = TabularDataLoaders.from_csv(
    path/'adult.csv', path=path, y_name="salary",
    cat_names=["workclass", "education", "marital-status", "occupation", "relationship"],  # 修正：marital-status 用连字符
    cont_names=['age', 'fnlwgt', 'education-num'],   # 连续数值列
    procs=[Categorify, FillMissing, Normalize]       # 预处理：类别编码 / 填缺失 / 标准化
)
dls.show_batch()


In [ ]:
# 训练表格模型，用 accuracy 作指标
learn = tabular_learner(dls, metrics=accuracy)
learn.fit_one_cycle(2)


In [ ]:
# —— 协同过滤 / 推荐示例（电影评分）——
from fastai.collab import *
path = untar_data(URLs.ML_SAMPLE)    # 修正：评分数据在 ML_SAMPLE，原来的 ADULT_SAMPLE 里没有 ratings.csv
dls = CollabDataLoaders.from_csv(path/'ratings.csv')


In [ ]:
# 查看一批 用户-电影-评分 数据
dls.show_batch()


In [ ]:
# 协同过滤模型，评分范围限定 0.5~5.5，微调 10 轮
learn = collab_learner(dls, y_range=(0.5, 5.5))
learn.fine_tune(10)


In [ ]:
# 展示预测评分 vs 真实评分
learn.show_results()
